In [2]:
## importing the library use
import pandas as pd
import seaborn as sns

In [ ]:
## loadint data sets
customers = pd.read_csv("../Data/European_Bank.csv")

In [4]:
customers.columns

Index(['Year', 'CustomerId', 'Surname', 'CreditScore', 'Geography', 'Gender',
       'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='str')

In [5]:
## in this data set
# year is 2025 which is same for all entries 
#     this has variance 0 does not affect model remove it
# CustomerID it has no impact on churn remove it because it can havily lift the models
# surname is also does not impact the model behavior on the financial data problem remove it
customers.drop(['Year','CustomerId','Surname'],inplace=True,axis=1) # droping the whole column

In [6]:
# seperating the columns for model creation
num_cols = ['CreditScore','Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary']

# columns have text data, greater then 2 different colums value value = ['Germany','France','spain']
one_hot_cols = ['Geography']

binary_cols = ['Gender']

In [7]:
# chnage the male -> 1 and female -> 0 because machine can only understand numbers

customers['Gender'] = customers['Gender'].apply(lambda x: 1 if x == "Male" else 0)

In [8]:
customers.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [9]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  str    
 2   Gender           10000 non-null  int64  
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(8), str(1)
memory usage: 918.1 KB


#### One-Hot Encoding : 
Use this for nominal data where categories have no inherent order or ranking (e.g., colors, cities). It creates a new binary column (1 or 0) for each unique category in the dataset
`in sort when we have multiple value which in column but value are independent of each other use One hot encoder`

In [10]:
from sklearn.preprocessing import OneHotEncoder

In [11]:
one_hot_encoder = OneHotEncoder(sparse_output=False) # sparse_output = False make the OneHotEncoder to produce the ndarray instead of sparse array
encoder_data = one_hot_encoder.fit_transform(customers[['Geography']])

#### Label encoder/ Ordinal encoder : 
use this for ordinal data where categories follow a natural order or rank ( e.g education level, satisfaction rating). It assings a unique sequential integer to each category based on its rank
`In sort when we have multiple value with in the column and they are related to each other then use the labell encoder`

In [ ]:
## now
# one_hot_encoder.get_feature_names_out(['Geography']) is just the columns name given by the one_hot_encoder
geography = pd.DataFrame(encoder_data,columns=one_hot_encoder.get_feature_names_out(['Geography']))

In [13]:
## merging the geography to the customer dataframe 
customers.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='str')

In [14]:
# extracting the usefull columns and from customer table
input_data = pd.concat( [customers[num_cols+['Gender']], geography],axis=1)
output_data = customers['Exited']

In [15]:
# spliting the data into train and test 
# train - 70% , validation - 10% and test - 20%
from sklearn.model_selection import train_test_split
x_train, x_remain, y_train, y_remain = train_test_split(input_data,output_data,test_size=0.3,random_state=42,stratify=output_data)

# now i have train data x_train -> input for train and y_train -> output of train
# spliting the remaining x and y to 10 and 20 validation and test partition
x_validate, x_test, y_validate, y_test = train_test_split(x_remain, y_remain, test_size=0.7,random_state=42,stratify=y_remain)

In [ ]:
# importing different model available
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [17]:
# defining the models i want to test
models = {
    "LogisticRegression" : LogisticRegression(max_iter=100),
    "RandomForestClassifier" : RandomForestClassifier(random_state=42),
    "XGBClassifier" : XGBClassifier(random_state=42,eval_metric="logloss")
}

In [18]:
# executing the models and finding the output
results = {}
for model_name, model in models.items():
    # training the model
    model.fit(x_train,y_train)

    # make predictions on validation
    predictions = model.predict(x_validate)

    results[model_name] = {
        "Accuracy": accuracy_score(y_validate,predictions),
        "Precision": precision_score(y_validate,predictions),
        "Recall": recall_score(y_validate, predictions),
        "F1-score": f1_score(y_validate,predictions)
    }

/home/nitish/Desktop/Study_Stuff/Machine_Learning/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [19]:
result_df = pd.DataFrame(results).T

In [20]:
model_info = {}
# removing the linearregression model it has poor performance
del models["LogisticRegression"]
# again checking the model output on the test data set
for model_name, model in models.items():
    x_input = pd.concat( [x_train, x_validate] )
    y_output = pd.concat( [y_train, y_validate] )
    model.fit(x_input, y_output)

    # predict on the testing data set
    pred = model.predict(x_test)

    model_info[model_name] = {
        "Accuracy": accuracy_score(y_test,pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test,pred),
        "f1-score": f1_score(y_test, pred)
    }

In [21]:
model_info = pd.DataFrame(model_info).T

In [22]:
model_info

,Accuracy,Precision,Recall,f1-score
RandomForestClassifier,0.860476,0.754717,0.467290,0.577201
XGBClassifier,0.849048,0.684385,0.481308,0.565158


In [23]:
result_df

,Accuracy,Precision,Recall,F1-score
LogisticRegression,0.793333,0.411765,0.038251,0.070000
RandomForestClassifier,0.847778,0.730000,0.398907,0.515901
XGBClassifier,0.841111,0.661290,0.448087,0.534202


### saving the created model to as the pickle

In [10]:
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier

In [28]:
customers.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='str')

In [ ]:
# seperating the columns
num_cols = ['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary']

one_hot_cols = ['Gender','Geography']

In [30]:
# creaing the preprocessor pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('OneHot_Cols',OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),one_hot_cols),
        ('Num_Cols',StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

In [31]:
# creating the final model
final_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=4
)

In [32]:
# connecting the engine and the model
production_pipeline = Pipeline(steps=[
    ('preprocessing',preprocessor),
    ('model', final_model)
])

In [34]:
# fitting all available data into the model
x = customers.drop(columns=['Exited'])
y = customers['Exited']

In [35]:
production_pipeline.fit(x,y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['CreditScore','Geography','Gender',...,'HasCrCard','IsActiveMember', 'EstimatedSalary']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('OneHot_Cols', ...), ('Num_Cols', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

In [36]:
# saving the model 
joblib.dump(production_pipeline,'../Models/churn_prediction_pipeline.pkl')
print("Successfully saved model")

Successfully saved model


In [24]:
final_model = XGBClassifier(
    random_state=42,
    eval_metric='logloss',
    scale_pos_weight=4
)

In [26]:
production_pipeline = Pipeline(steps=[
    ("processing",preprocessor),
    ('model',final_model)
])

NameError: name 'preprocessor' is not defined

In [1]:
# importing the pretrained model to check accuracy for different values
import joblib

In [2]:
model = joblib.load("../Models/churn_prediction_model.pkl")

In [3]:
# loading the data set 
import pandas as pd
df = pd.read_csv("../Data/European_Bank.csv")

In [6]:
# deleting the extra columns
df.drop(['Year','CustomerId'],inplace=True,axis=1)

In [8]:
df.drop(['Surname'],inplace=True, axis=1)

In [9]:
df.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='str')

In [12]:
# creaing the preprocessor pipeline
num_cols = ['CreditScore','Age','Tenure','Balance','NumOfProducts','HasCrCard','IsActiveMember','EstimatedSalary']

one_hot_cols = ['Gender','Geography']
preprocessor = ColumnTransformer(
    transformers=[
        ('OneHot_Cols',OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'),one_hot_cols),
        ('Num_Cols',StandardScaler(), num_cols)
    ],
    remainder='passthrough'
)

In [13]:
production_pipeline = Pipeline(steps=[
    ("processing",preprocessor),
    ('model',model)
])

In [14]:
new_df = df.drop(['Exited'],axis=1)

In [15]:
exited = df['Exited']

In [20]:
model.predict(new_df)

ValueError: could not convert string to float: 'Female'